# 01 · Preprocess ATD & Smart Tagging

**Tujuan.** Mengubah data mentah menjadi array per **frame video** yang bersih dan dalam **meter** — satu sumber
kebenaran untuk semua tahap berikutnya. Tidak ada keputusan H/A/D di sini; hanya membaca, membersihkan,
dan mengukur kualitas data.

| | |
|---|---|
| **Input** | `tactical/BAR-ATH_tactical_atd.xml`, `_atd.txt`, `_smart_tagging.xml` |
| **Output** | `cache/prep.npz`, `cache/prep_meta.json`, `cache/gaps.csv`, `cache/tags.csv`, laporan `cache/reports/01_preprocess.*` |
| **Waktu** | ± 1 menit |
| **Privasi** | hanya mencetak agregat; semua output di `cache/` (gitignored) |

### Isi `cache/prep.npz` (semua diindeks frame video `0…199318`, **bukan** `frame_id`)

| Kunci | Bentuk | Isi |
|---|---|---|
| `players` | (N, 46, 3) float32 | `x_m`, `y_m`, kecepatan m/s per slot; `NaN` = tidak terlacak |
| `slot_is_home` | (46,) bool | slot → tim kandang (dari XML) |
| `n_home`, `n_away` | (N,) int8 | jumlah slot terlacak per tim |
| `ball_raw` | (N, 2) float32 | posisi bola (m), `NaN` bila hilang |
| `ball_filled` | (N, 2) float32 | `ball_raw` + interpolasi linear celah pendek yang masuk akal |
| `ball_src` | (N,) int8 | 0 = hilang, 1 = teramati, 2 = interpolasi |
| `ball_speed` | (N,) float32 | m/s dari selisih posisi antar frame (kolom kecepatan bola di ATD hampir selalu kosong) |
| `ball_oob_m` | (N,) float32 | jarak bola di luar garis lapangan (0 = di dalam, `NaN` = hilang) |
| `ball_jump` | (N,) bool | perpindahan antar frame berurutan yang mustahil secara fisik |
| `half` | (N,) int8 | 0 = tidak dinilai, 1, 2 |
| `graded` | (N,) bool | frame yang dinilai |
| `dead_prior` | (N,) bool | babak 2 sebelum kick-off — pasti `D` (FAQ.md) |
| `attack_dir` | (N,) int8 | +1 bila tim **H** menyerang ke arah +x, −1 sebaliknya, 0 di luar babak |

Koordinat: `x` ∈ [0, 105,3] m sepanjang lapangan, `y` ∈ [0, 68] m.

In [ ]:
%run ./common.ipynb

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `MAX_GAP` | 25 frame (1 s) | celah lebih panjang dari ini tidak bisa diisi garis lurus dengan jujur — bola bisa dioper, memantul, keluar |
| `MAX_GAP_SPEED` | 40 m/s | kecepatan rata-rata yang dibutuhkan untuk menyeberangi celah; tendangan terkeras ± 35–40 m/s, di atas itu kemungkinan besar dua deteksi berbeda |
| `JUMP_M` | 3,0 m/frame | = 75 m/s, mustahil untuk bola; hanya **ditandai**, tidak dihapus, agar tahap berikut yang memutuskan |

In [ ]:
MAX_GAP = 25
MAX_GAP_SPEED = 40.0
JUMP_M = 3.0

In [ ]:
rep = Report("01_preprocess")
CHECKS = []


def check(name, ok, detail="", critical=True):
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if critical else "WARN")
    CHECKS.append({"cek": name, "status": status, "kritis": critical, "keterangan": detail})
    rep("[{}] {}{}".format(status, name, " — " + detail if detail else ""))
    return ok

## 1. Metadata: tim per slot & arah serang

Dibaca dengan `parse_atd_meta()` (`common.ipynb` §7) dan sudah dicek di `00_setup_cek_data`. Di sini cukup
`assert` singkat, lalu diturunkan dua hal yang dipakai terus:

- `SLOT_IS_HOME` — tim tiap slot.
- `ATTACK_DIR` per frame — arah serang tim kandang. Tim menukar sisi di babak 2, jadi tandanya dibalik.
  Fitur "maju ke gawang lawan" di Fase 2 bergantung pada tanda ini, jadi di bagian 4 dicek ulang dengan posisi
  pemain sesaat sebelum kick-off.

In [ ]:
meta = parse_atd_meta()
SLOT_IS_HOME = meta["slot_is_home"]
N_SLOTS = len(SLOT_IS_HOME)
assert meta["slot_ids"] == meta["channel_order"] and meta["local_team_id"] == meta["home_id"]
assert meta["attack_h1"] in ("left_to_right", "right_to_left")

H_SIGN_H1 = 1 if meta["attack_h1"] == "left_to_right" else -1
ATTACK_DIR = np.zeros(N_FRAMES, dtype=np.int8)
ATTACK_DIR[HALF == 1] = H_SIGN_H1
ATTACK_DIR[HALF == 2] = -H_SIGN_H1

rep("H = {} ({} slot) | A = {} ({} slot)".format(
    meta["home_name"], int(SLOT_IS_HOME.sum()), meta["away_name"], int((~SLOT_IS_HOME).sum())))
rep("H menyerang ke {} di babak 1, ke {} di babak 2".format(
    "+x" if H_SIGN_H1 > 0 else "-x", "-x" if H_SIGN_H1 > 0 else "+x"))

## 2. Membaca ATD penuh

Satu baris = `frame_id:grup;grup;…:bola`, grup = `x,y,kecepatan` dalam 0–1. Seluruh grup pemain di satu baris
diubah ke array sekaligus (`"NaN"` otomatis menjadi `nan`), lalu dibentuk ulang menjadi (46, 3).

Baris ke-*i* file = `frame_id` *i*+1 = **frame video *i*** (diverifikasi di notebook 00), jadi array langsung
diindeks dengan frame video.

In [ ]:
def load_atd(path, n_slots):
    ids = np.zeros(N_FRAMES, dtype=np.int64)
    players = np.full((N_FRAMES, n_slots, 3), np.nan, dtype=np.float32)
    ball = np.full((N_FRAMES, 3), np.nan, dtype=np.float32)
    with open(path, encoding="utf-8") as handle:
        for i, line in enumerate(handle):
            frame, groups, b = line.rstrip("\r\n").split(":")
            ids[i] = int(frame)
            players[i] = np.array(groups.replace(";", ",").split(","), dtype=np.float32).reshape(n_slots, 3)
            ball[i] = np.array(b.split(","), dtype=np.float32)
    return ids, players, ball


with timer("parse ATD"):
    atd_ids, players, ball = load_atd(require_asset("atd_raw"), N_SLOTS)

check("frame_id = frame video + 1 di semua baris", (atd_ids == np.arange(1, N_FRAMES + 1)).all())

## 3. Konversi ke meter

Tracking dinormalisasi 0–1, padahal lapangan tidak persegi (105,3 × 68 m). Tanpa konversi, jarak di sumbu
pendek mengecil ± 35% dan "pemain terdekat" bisa salah — terutama dekat garis samping. Kecepatan pemain di ATD
sudah dalam m/s.

Kolom kecepatan bola (`s`) hampir selalu kosong, jadi kecepatan bola dihitung ulang dari posisi di bagian 6.

In [ ]:
players[..., 0] *= PITCH_L
players[..., 1] *= PITCH_W
ball_raw = (ball[:, :2] * np.array([PITCH_L, PITCH_W], dtype=np.float32)).astype(np.float32)

finite = np.isfinite(ball_raw[:, 0])
speed_col_empty = float(np.isnan(ball[finite, 2]).mean())
rep("kolom kecepatan bola kosong pada {:.1f}% frame yang bolanya ada".format(100 * speed_col_empty))
check("pemain & bola dalam rentang lapangan wajar (±10 m)",
      np.nanmin(players[..., 0]) > -10 and np.nanmax(players[..., 0]) < PITCH_L + 10
      and np.nanmin(players[..., 1]) > -10 and np.nanmax(players[..., 1]) < PITCH_W + 10,
      "x [{:.1f}, {:.1f}] y [{:.1f}, {:.1f}]".format(np.nanmin(players[..., 0]), np.nanmax(players[..., 0]),
                                                      np.nanmin(players[..., 1]), np.nanmax(players[..., 1])),
      critical=False)

## 4. Topeng frame & cek arah serang

- `half`, `graded` diambil dari `common` (`HALF`, `GRADED`).
- `dead_prior`: frame 110.808–113.741 — babak 2 sudah dinilai tetapi belum kick-off, jadi pasti `D` (FAQ.md).
- **Cek arah serang dengan data:** sesaat sebelum kick-off setiap tim berdiri di separuh lapangannya sendiri.
  Bila H menyerang ke +x, sentroid H harus di sebelah **kiri** sentroid A. Cek ini memastikan tanda
  `attack_dir` tidak terbalik — kesalahan yang diam-diam akan membalik semua fitur arah.

In [ ]:
dead_prior = np.zeros(N_FRAMES, dtype=bool)
dead_prior[HALVES[2][0]:KICKOFF[2]] = True
rep("frame dinilai {} | prior D babak 2: {} frame".format(N_GRADED, int(dead_prior.sum())))

with np.errstate(all="ignore"):
    for h, ko in KICKOFF.items():
        window = slice(ko - FPS, ko)
        hx = np.nanmean(players[window][:, SLOT_IS_HOME, 0])
        ax = np.nanmean(players[window][:, ~SLOT_IS_HOME, 0])
        sign = ATTACK_DIR[ko]
        ok = (hx < ax) if sign > 0 else (hx > ax)
        check("arah serang babak {} konsisten dengan posisi sebelum kick-off".format(h), ok,
              "sentroid x H {:.1f} m vs A {:.1f} m, H menyerang {}".format(hx, ax, "+x" if sign > 0 else "-x"))

## 5. Kualitas tracking pemain

Seharusnya ± 11 slot terlacak per tim. Lebih dari 11 = tracker menduplikasi seseorang; kurang dari 8 = banyak
pemain hilang (biasanya di luar kamera atau berkerumun). Angka ini menentukan seberapa bisa dipercaya fitur bentuk
tim di Fase 2, dan `n_home`/`n_away` disimpan supaya fitur itu bisa menimbang dirinya sendiri.

In [ ]:
present = np.isfinite(players[..., 0])
n_home = present[:, SLOT_IS_HOME].sum(1).astype(np.int8)
n_away = present[:, ~SLOT_IS_HOME].sum(1).astype(np.int8)

rows = []
for lab, n in (("H", n_home), ("A", n_away)):
    g = n[GRADED]
    rows.append({"tim": lab, "rata-rata/frame": round(float(g.mean()), 2), "median": int(np.median(g)),
                 ">11 (duplikat?)": int((g > 11).sum()), "<8": int((g < 8).sum())})
player_q = pd.DataFrame(rows).set_index("tim")
rep.table(player_q, "slot terlacak per tim (frame dinilai):")
never = [meta["slot_ids"][k] for k in np.flatnonzero(~present.any(0))]
rep("slot yang tidak pernah terisi (id): {}".format(never))

## 6. Bola: kehadiran, keluar lapangan, kecepatan, lompatan

- **Kehadiran** per babak.
- **`ball_oob_m`** — jarak ke luar garis lapangan. Bola yang benar-benar keluar adalah bukti kuat bola mati,
  tetapi tracker sering kehilangan bola justru saat keluar, jadi bukti ini jarang tersedia.
- **`ball_speed`** — selisih posisi antar frame berurutan × 25. Masih berisik; tahap berikut yang menghaluskannya.
- **`ball_jump`** — perpindahan > `JUMP_M` dalam satu frame; tanda deteksi salah atau pergantian objek.

In [ ]:
for h, (a, b) in HALVES.items():
    rep("bola ada di babak {}: {:.2f}%".format(h, 100 * finite[a:b + 1].mean()))
rep("bola ada: semua frame {:.2f}% | frame dinilai {:.2f}% ({} / {})".format(
    100 * finite.mean(), 100 * finite[GRADED].mean(), int(finite[GRADED].sum()), N_GRADED))

x, y = ball_raw[:, 0], ball_raw[:, 1]
dx = np.maximum.reduce([-x, x - PITCH_L, np.zeros_like(x)])
dy = np.maximum.reduce([-y, y - PITCH_W, np.zeros_like(y)])
ball_oob_m = np.where(finite, np.hypot(dx, dy), np.nan).astype(np.float32)
g_oob = ball_oob_m[GRADED & finite]
rep("bola di luar garis (frame dinilai, teramati): >0 m {} ({:.1f}%) | >5 m {}".format(
    int((g_oob > 0).sum()), 100 * (g_oob > 0).mean(), int((g_oob > 5).sum())))

step = np.full(N_FRAMES, np.nan, dtype=np.float32)
both = finite[1:] & finite[:-1]
step[1:][both] = np.hypot(*(ball_raw[1:][both] - ball_raw[:-1][both]).T)
ball_speed = (step * FPS).astype(np.float32)
ball_jump = np.zeros(N_FRAMES, dtype=bool)
ball_jump[1:][both] = step[1:][both] > JUMP_M
rep("lompatan > {:.1f} m antar frame (frame dinilai): {}".format(JUMP_M, int((ball_jump & GRADED).sum())))
rep("kecepatan bola antar frame (dinilai, teramati): median {:.1f} m/s, p95 {:.1f} m/s".format(
    float(np.nanmedian(ball_speed[GRADED])), float(np.nanpercentile(ball_speed[GRADED], 95))))

## 7. Celah bola & interpolasi

Celah = run frame tanpa bola di dalam satu babak. Untuk tiap celah dicatat panjangnya, apakah ada bola
sebelum/sesudahnya, dan jarak bola terakhir-terlihat → muncul-lagi.

**Aturan interpolasi:** celah ≤ `MAX_GAP` frame, ada bola di kedua sisi, dan kecepatan rata-rata yang
dibutuhkan ≤ `MAX_GAP_SPEED` → isi dengan garis lurus. Selain itu dibiarkan hilang; tahap berikut harus bekerja
tanpa bola. Hasilnya `ball_filled` dan `ball_src`, plus `cache/gaps.csv` (satu baris per celah).

Distribusi panjang celah adalah fakta terpenting dataset ini: bila hampir semua frame hilang berada di celah
panjang, interpolasi nyaris tidak membantu, dan possession harus disimpulkan dari pemain.

In [ ]:
gap_rows = []
for h, (a, b) in HALVES.items():
    s, e = runs(~finite[a:b + 1])
    for si, ei in zip(s, e):
        gap_rows.append({"half": h, "start": a + si, "end_excl": a + ei, "length": ei - si,
                         "has_left": si > 0, "has_right": a + ei <= b})
gaps = pd.DataFrame(gap_rows)
gaps["dist_m"] = np.nan
both_sides = gaps.has_left & gaps.has_right
gaps.loc[both_sides, "dist_m"] = [float(np.hypot(*(ball_raw[r.end_excl] - ball_raw[r.start - 1])))
                                   for r in gaps[both_sides].itertuples()]
gaps["need_speed"] = gaps.dist_m / ((gaps.length + 1) / FPS)
gaps["interpolated"] = both_sides & (gaps.length <= MAX_GAP) & (gaps.need_speed <= MAX_GAP_SPEED)

ball_filled = ball_raw.copy()
ball_src = finite.astype(np.int8)
for r in gaps[gaps.interpolated].itertuples():
    t = (np.arange(1, r.length + 1) / (r.length + 1))[:, None]
    p0, p1 = ball_raw[r.start - 1], ball_raw[r.end_excl]
    ball_filled[r.start:r.end_excl] = p0 + (p1 - p0) * t
    ball_src[r.start:r.end_excl] = 2

missing = int(gaps.length.sum())
rep("{} celah, {} frame hilang | median {:.0f}, p90 {:.0f}, maks {} frame ({:.1f} s)".format(
    len(gaps), missing, gaps.length.median(), gaps.length.quantile(0.9), int(gaps.length.max()),
    gaps.length.max() / FPS))
bins = [(1, 25), (26, 125), (126, 500), (501, 2000), (2001, 10 ** 9)]
dist = pd.DataFrame([{"panjang (frame)": "{}-{}".format(lo, hi) if hi < 10 ** 9 else ">{}".format(lo - 1),
                      "celah": int(((gaps.length >= lo) & (gaps.length <= hi)).sum()),
                      "frame": int(gaps.length[(gaps.length >= lo) & (gaps.length <= hi)].sum())}
                     for lo, hi in bins]).set_index("panjang (frame)")
dist["% frame hilang"] = (100 * dist.frame / max(missing, 1)).round(1)
rep.table(dist, "distribusi panjang celah:")
n_int, f_int = int(gaps.interpolated.sum()), int(gaps.length[gaps.interpolated].sum())
rep("diinterpolasi: {} celah, {} frame = {:.2f}% dari yang hilang".format(n_int, f_int, 100 * f_int / max(missing, 1)))
long_d = gaps.dist_m[gaps.length > MAX_GAP].dropna()
rep("celah panjang: jarak bola hilang → muncul lagi median {:.1f} m, p90 {:.1f} m".format(
    long_d.median(), long_d.quantile(0.9)))

## 8. Smart Tagging dalam frame video

Dibaca dengan `load_smart_tagging()` (`common.ipynb` §7). Diringkas per kode dan, untuk `SET PIECES`, per jenis
— termasuk **durasi jendela tag**. Bila durasinya seragam, jendela itu bukan durasi bola mati yang sebenarnya,
dan yang bermakna hanya titik awal/akhirnya.

In [ ]:
tags = load_smart_tagging(meta)
rep("instans {} | sisi Team: {}".format(len(tags), tags.side.value_counts().to_dict()))
check("semua nilai Team dikenali", not (tags.side == "?").any())

per_code = (tags.assign(n=1, dur_s=(tags.end_f - tags.start_f) / FPS)
            .groupby("code").agg(n=("n", "sum"), H=("side", lambda s: int((s == "H").sum())),
                                 A=("side", lambda s: int((s == "A").sum())),
                                 NA=("side", lambda s: int((s == "NA").sum())),
                                 durasi_min_s=("dur_s", "min"), durasi_maks_s=("dur_s", "max"))
            .sort_values("n", ascending=False))
rep.table(per_code, "per kode:")

sp = tags[tags.code == "SET PIECES"]
per_type = (sp.assign(n=1, dur_s=(sp.end_f - sp.start_f) / FPS)
            .groupby("Type").agg(n=("n", "sum"), H=("side", lambda s: int((s == "H").sum())),
                                 A=("side", lambda s: int((s == "A").sum())),
                                 NA=("side", lambda s: int((s == "NA").sum())),
                                 durasi_min_s=("dur_s", "min"), durasi_maks_s=("dur_s", "max")))
rep.table(per_type, "SET PIECES per jenis:")

## 9. Simpan

Array disimpan tanpa kompresi (memuat cepat). `prep_meta.json` mencatat metadata dan parameter yang
menghasilkan file ini, supaya setiap artefak bisa dilacak asal-usulnya.

In [ ]:
np.savez(CACHE / "prep.npz", players=players, slot_is_home=SLOT_IS_HOME, n_home=n_home, n_away=n_away,
         ball_raw=ball_raw, ball_filled=ball_filled, ball_src=ball_src, ball_speed=ball_speed,
         ball_oob_m=ball_oob_m, ball_jump=ball_jump, half=HALF, graded=GRADED,
         dead_prior=dead_prior, attack_dir=ATTACK_DIR)
gaps.to_csv(CACHE / "gaps.csv", index=False)
tags.to_csv(CACHE / "tags.csv", index=False)
prep_meta = {"home_name": meta["home_name"], "away_name": meta["away_name"], "slot_ids": meta["slot_ids"],
             "attack_h1": meta["attack_h1"], "params": {"MAX_GAP": MAX_GAP, "MAX_GAP_SPEED": MAX_GAP_SPEED,
                                                        "JUMP_M": JUMP_M}}
(CACHE / "prep_meta.json").write_text(json.dumps(prep_meta, indent=2), encoding="utf-8")

z = load_prep()
check("prep.npz bisa dimuat ulang dengan semua kunci", len(z) == 14, str(sorted(z)))
check("bentuk array benar", z["players"].shape == (N_FRAMES, N_SLOTS, 3) and z["ball_raw"].shape == (N_FRAMES, 2))
check("ball_filled hanya terisi di frame teramati/interpolasi",
      (np.isfinite(z["ball_filled"][:, 0]) == (z["ball_src"] > 0)).all())
check("tags.csv bisa dimuat ulang", len(load_tags()) == len(tags))

## 10. Cek regresi terhadap versi uji coba

Skrip uji coba lama (sudah dihapus) menghasilkan angka-angka di bawah dengan logika yang sama. Notebook ini harus
menghasilkan angka **identik** — kalau berbeda, berarti ada perubahan perilaku yang harus dijelaskan.
Tingkatnya peringatan, karena versi lama bukan kebenaran mutlak.

In [ ]:
expected = {
    "bola ada di frame dinilai": (int(finite[GRADED].sum()), 94214),
    "slot H": (int(SLOT_IS_HOME.sum()), 24),
    "slot tak pernah terisi": (never, [44]),
    "H >11 slot (frame dinilai)": (int(player_q.loc["H", ">11 (duplikat?)"]), 12921),
    "A >11 slot (frame dinilai)": (int(player_q.loc["A", ">11 (duplikat?)"]), 7529),
    "H <8 slot (frame dinilai)": (int(player_q.loc["H", "<8"]), 10812),
    "A <8 slot (frame dinilai)": (int(player_q.loc["A", "<8"]), 9105),
    "bola di luar garis >0 m": (int((g_oob > 0).sum()), 3949),
    "bola di luar garis >5 m": (int((g_oob > 5).sum()), 981),
    "lompatan > 3 m": (int((ball_jump & GRADED).sum()), 13),
    "jumlah celah": (len(gaps), 150),
    "frame hilang": (missing, 64643),
    "celah diinterpolasi": (n_int, 5),
    "frame diinterpolasi": (f_int, 41),
}
for name, (got, want) in expected.items():
    check("regresi: " + name, got == want, "{} vs {}".format(got, want), critical=False)

## 11. Ringkasan

In [ ]:
summary_df = pd.DataFrame(CHECKS)
n_fail = int((summary_df.status == "FAIL").sum())
rep("\nPASS {} | WARN {} | FAIL {}".format(int((summary_df.status == "PASS").sum()),
                                         int((summary_df.status == "WARN").sum()), n_fail))
rep.save({"checks": CHECKS, "player_quality": player_q.reset_index().to_dict("records"),
          "gap_distribution": dist.reset_index().to_dict("records"),
          "ball_present_graded": int(finite[GRADED].sum()), "n_gaps": len(gaps), "missing_frames": missing,
          "interpolated": {"gaps": n_int, "frames": f_int}})
assert n_fail == 0, "{} cek kritis gagal".format(n_fail)

## Ringkasan & keputusan

**Fakta dari data yang membentuk desain tahap berikutnya**

1. **Bola hilang dalam blok panjang.** Hampir semua frame hilang ada di celah > 1 s (lihat distribusi di bagian 7);
   interpolasi hanya menutup sebagian sangat kecil. **Keputusan:** jangan bergantung pada posisi bola. Fase 2
   memakai fitur bentuk/gerak tim yang selalu tersedia, dan tahap bola mati memakai "bola hilang" sebagai bukti
   lemah, bukan sebagai label.
2. **Tracking pemain tidak sempurna.** Banyak frame dengan > 11 atau < 8 slot per tim. **Keputusan:** fitur tim
   berbasis statistik yang tahan (median, kuantil) dan ditimbang dengan `n_home`/`n_away`, bukan per pemain.
3. **Kolom kecepatan bola di ATD hampir selalu kosong.** **Keputusan:** kecepatan dihitung dari posisi.
4. **Jendela tag `SET PIECES` berdurasi seragam per jenis** (lihat bagian 8). **Keputusan:** jendela tag tidak
   dipakai sebagai durasi bola mati; hanya titik awal/akhirnya sebagai petunjuk waktu restart.
5. **Throw-in dan kick-off tidak punya tim di tag.** **Keputusan:** tim pengambil ditebak dari geometri (Fase 3.2).
6. **Arah serang terverifikasi** dengan posisi pemain sebelum kedua kick-off.

**Berikutnya:** `02_eda_gt_publik.ipynb` — memahami apa yang GT publik sebut `D` dan kapan ia mengganti `H`/`A`.